# Rolling-origin validation — is the result stable, or was September lucky?

`merchant_ranking_hybrid.ipynb` and `learned_weights.ipynb` both end on a single evaluation month,
September 2022. That is honest, but it is one month: a single large merchant having an unusual
September moves the numbers more than it should.

This notebook repeats the same test across **four windows that walk forward through time**, each
with a **three-month** target rather than one. If the ranking's advantage survives all four, it is
not an artefact of September. If it does not, that is the more valuable finding, and better that we
report it than that it is asked about in the presentation.

**What the data allows.** Transactions run 28 Feb 2021 – 26 Oct 2022, but February holds a single
day and October stops on the 26th. Only **March 2021 – September 2022** are complete months, so
every window below sits inside that range and no partial month is ever used as a target.

| Fold | Features built from | Target (3 months) | Weights learned on |
| --- | --- | --- | --- |
| **A** | Mar – Sep 2021 | Oct – Dec 2021 | — (no earlier window) |
| **B** | Mar – Dec 2021 | Jan – Mar 2022 | fold A's pair |
| **C** | Mar 2021 – Mar 2022 | Apr – Jun 2022 | fold B's pair |
| **FINAL** | Mar 2021 – Jun 2022 | Jul – Sep 2022 | fold C's pair |

**The rule that keeps this clean:** a fold's model may only be fitted on a window whose target ends
*before* that fold's target begins. Fold A has no such window, so no learned model is scored there —
only the hand weights and the baseline, which need no fitting.

The features expand as time passes, which is how the firm would actually work: every month that
arrives is one more month of evidence.

September is reused across exploratory evaluations. These tables do not establish an independent model-selection holdout. Frozen weights are not overwritten by these fits.


In [1]:
K = 300
WEIGHTS_HYBRID = {
    "estimated_bnpl_revenue": .70, "log_value_growth": .02,
    "transactions_per_customer": 0., "repeat_rate": 0., "stability": 0.,
    "active_month_ratio": .18, "customer_spread": .05,
    "customer_income": .03, "customer_irsad": .02,
}
WEIGHTS_HAND = {
    "estimated_bnpl_revenue": .70, "log_value_growth": .08,
    "transactions_per_customer": .04, "repeat_rate": .03, "stability": .03,
    "active_month_ratio": .02, "customer_spread": .05,
    "customer_income": .03, "customer_irsad": .02,
}


# Resolve imports by repository contents, regardless of notebook location.
import sys
from pathlib import Path
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "scripts" / "process_cleaning.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the project repository")
if str(PROJECT_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))


from pathlib import Path

import numpy as np
import pandas as pd
from pyspark.sql import SparkSession, Window, functions as F

ROOT = PROJECT_ROOT
PROCESSED = ROOT / "data" / "tables" / "processed"

FOLDS = [
    {"name": "A",     "feat": ("2021-03-01", "2021-09-30"), "months": 7,
     "label": ("2021-10-01", "2021-12-31")},
    {"name": "B",     "feat": ("2021-03-01", "2021-12-31"), "months": 10,
     "label": ("2022-01-01", "2022-03-31")},
    {"name": "C",     "feat": ("2021-03-01", "2022-03-31"), "months": 13,
     "label": ("2022-04-01", "2022-06-30")},
    {"name": "FINAL", "feat": ("2021-03-01", "2022-06-30"), "months": 16,
     "label": ("2022-07-01", "2022-09-30")},
]

spark = (
    SparkSession.builder.config("spark.sql.session.timeZone", "Etc/UTC").appName("MAST30034 Project 2 - Rolling validation")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

transactions = spark.read.parquet(str(PROCESSED / "transactions_clean"))
merchants = spark.read.parquet(str(PROCESSED / "merchants_clean"))
consumers = spark.read.parquet(str(PROCESSED / "consumers_sa2"))

print("folds:")
for f in FOLDS:
    print(f"  {f['name']:<6} features {f['feat'][0]} to {f['feat'][1]} "
          f"({f['months']} months)  ->  target {f['label'][0]} to {f['label'][1]}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/13 23:26:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


folds:
  A      features 2021-03-01 to 2021-09-30 (7 months)  ->  target 2021-10-01 to 2021-12-31
  B      features 2021-03-01 to 2021-12-31 (10 months)  ->  target 2022-01-01 to 2022-03-31
  C      features 2021-03-01 to 2022-03-31 (13 months)  ->  target 2022-04-01 to 2022-06-30
  FINAL  features 2021-03-01 to 2022-06-30 (16 months)  ->  target 2022-07-01 to 2022-09-30


## 1. Features rebuilt per window

All feature definitions come from the calendar-window function below: calendar growth blocks, zero-filled monthly sample volatility, and customer profiles restricted to each feature window. Production features cannot be reused for earlier predictions.


In [2]:
import pandas as pd
from pyspark.sql import functions as F

def _build_window_features(
    tx,
    months_in_window,
    early_start,
    early_end_exclusive,
    recent_start,
    recent_end_exclusive,
    window_start,
    window_end,
    *, merchants, consumers,
):
    spark = tx.sparkSession
    tx = tx.filter((F.col("order_datetime") >= F.lit(window_start)) &
                   (F.col("order_datetime") < F.date_add(F.to_date(F.lit(window_end)), 1)))
    core = (
        tx.groupBy("merchant_abn")
        .agg(
            F.sum("dollar_value").alias("gmv"),
            F.count("*").alias("transaction_count"),
            F.countDistinct("user_id").alias("unique_customers"),
            F.countDistinct(F.trunc("order_datetime", "month")).alias("active_months"),
        )
        .withColumn(
            "transactions_per_customer",
            F.col("transaction_count") / F.col("unique_customers"),
        )
        .withColumn(
            "active_month_ratio",
            F.col("active_months") / F.lit(months_in_window),
        )
    )

    per_cust = (
        tx.groupBy("merchant_abn", "user_id")
        .agg(
            F.sum("dollar_value").alias("cust_gmv"),
            F.count("*").alias("cust_orders"),
        )
    )

    conc = (
        per_cust.groupBy("merchant_abn")
        .agg(
            (
                F.sum(F.when(F.col("cust_orders") > 1, 1).otherwise(0))
                / F.count("*")
            ).alias("repeat_rate"),
            F.sum(F.pow("cust_gmv", 2)).alias("_sq"),
            F.sum("cust_gmv").alias("_tot"),
        )
        .withColumn("customer_hhi", F.col("_sq") / F.pow(F.col("_tot"), 2))
        .drop("_sq", "_tot")
    )

    monthly = (
        tx.withColumn("month", F.trunc("order_datetime", "month"))
        .groupBy("merchant_abn", "month")
        .agg(F.sum("dollar_value").alias("monthly_gmv"))
    )

    month_starts = [
        (d.date(),)
        for d in pd.date_range(
            start=pd.Timestamp(window_start).replace(day=1),
            end=pd.Timestamp(window_end).replace(day=1),
            freq="MS",
        )
    ]
    complete_months = spark.createDataFrame(month_starts, ["month"])
    merchant_ids = tx.select("merchant_abn").distinct()

    monthly_complete = (
        merchant_ids.crossJoin(complete_months)
        .join(monthly, ["merchant_abn", "month"], "left")
        .fillna({"monthly_gmv": 0.0})
    )

    stab = (
        monthly_complete.groupBy("merchant_abn")
        .agg(
            F.avg("monthly_gmv").alias("mean_monthly_value"),
            F.stddev("monthly_gmv").alias("sd_monthly_value"),
        )
        .withColumn(
            "value_cv",
            F.when(
                F.col("mean_monthly_value") > 0,
                F.col("sd_monthly_value") / F.col("mean_monthly_value"),
            ),
        )
    )

    def three_month_block(start, end_exclusive, prefix):
        return (
            tx.filter(
                (F.col("order_datetime") >= F.lit(start))
                & (F.col("order_datetime") < F.lit(end_exclusive))
            )
            .groupBy("merchant_abn")
            .agg(
                F.sum("dollar_value").alias(f"{prefix}_3m_value"),
                F.countDistinct(F.trunc("order_datetime", "month")).alias(
                    f"{prefix}_active_months"
                ),
            )
        )

    early = three_month_block(early_start, early_end_exclusive, "early")
    recent = three_month_block(recent_start, recent_end_exclusive, "recent")

    growth = (
        early.join(recent, "merchant_abn", "full")
        .fillna(
            {
                "early_3m_value": 0.0,
                "recent_3m_value": 0.0,
                "early_active_months": 0,
                "recent_active_months": 0,
            }
        )
        .withColumn(
            "log_value_growth",
            F.when(
                (F.col("early_active_months") >= 2)
                & (F.col("recent_active_months") >= 2),
                F.log1p("recent_3m_value") - F.log1p("early_3m_value"),
            ),
        )
    )

    cp = per_cust.join(
        consumers.select(
            "user_id",
            "state",
            "sa2_code",
            "sa2_share",
            "abs_median_household_income_weekly",
            "seifa_irsad_decile",
        ),
        on="user_id",
        how="left",
    )

    has_income = F.col("abs_median_household_income_weekly").isNotNull()
    has_irsad = F.col("seifa_irsad_decile").isNotNull()
    wt = F.coalesce(F.col("sa2_share"), F.lit(1.0))

    base = (
        cp.groupBy("merchant_abn")
        .agg(
            (
                F.sum(
                    F.when(
                        has_income,
                        F.col("abs_median_household_income_weekly") * wt,
                    )
                )
                / F.sum(F.when(has_income, wt))
            ).alias("customer_income"),
            (
                F.sum(F.when(has_irsad, F.col("seifa_irsad_decile") * wt))
                / F.sum(F.when(has_irsad, wt))
            ).alias("customer_irsad"),
            F.avg(F.when(has_income, 1.0).otherwise(0.0)).alias(
                "pct_customers_with_abs"
            ),
        )
    )

    return (
        core.join(conc, "merchant_abn", "left")
        .join(stab.select("merchant_abn", "value_cv"), "merchant_abn", "left")
        .join(growth.select("merchant_abn", "log_value_growth"), "merchant_abn", "left")
        .join(base, "merchant_abn", "left")
        .join(
            merchants.select("merchant_abn", "name", "take_rate"),
            "merchant_abn",
            "left",
        )
        .withColumn(
            "estimated_bnpl_revenue",
            F.col("gmv") * F.col("take_rate") / 100.0,
        )
    )



def window_features(tx, merchants, consumers, window_start, window_end,
                    months_in_window=None, early_start=None, early_end_exclusive=None,
                    recent_start=None, recent_end_exclusive=None):
    start, end = pd.Timestamp(window_start), pd.Timestamp(window_end)
    if start.day != 1 or end != end + pd.offsets.MonthEnd(0) or end < start:
        raise ValueError("Feature windows must contain complete calendar months")
    months = (end.year - start.year) * 12 + end.month - start.month + 1
    if months_in_window is not None and months_in_window != months:
        raise ValueError("Month count must match the complete calendar window")
    return _build_window_features(
        tx, months, early_start or str(start.date()),
        early_end_exclusive or str((start + pd.DateOffset(months=3)).date()),
        recent_start or str((end.replace(day=1) - pd.DateOffset(months=2)).date()),
        recent_end_exclusive or str((end.replace(day=1) + pd.DateOffset(months=1)).date()),
        str(start.date()), str(end.date()), merchants=merchants, consumers=consumers,
    )


def build_features(tx, months_in_window, window_start, window_end):
    return window_features(tx, merchants, consumers, window_start, window_end,
                           months_in_window)


def realised(tx):
    return (tx.groupBy("merchant_abn")
              .agg(F.sum("dollar_value").alias("label_gmv"),
                   F.count("*").alias("label_transactions"))
              .join(merchants.select("merchant_abn", "take_rate"), "merchant_abn", "left")
              .withColumn("realised_revenue", F.col("label_gmv") * F.col("take_rate") / 100.0)
              .select("merchant_abn", "label_transactions", "realised_revenue"))


frames = {}
for fold in FOLDS:
    feats = build_features(
        transactions.filter(F.col("order_datetime").between(*fold["feat"])), fold["months"], *fold["feat"]
    ).filter(F.col("take_rate").isNotNull())
    lab = realised(transactions.filter(F.col("order_datetime").between(*fold["label"])))
    d = feats.join(lab, "merchant_abn", "left").toPandas()
    d[["label_transactions", "realised_revenue"]] = \
        d[["label_transactions", "realised_revenue"]].fillna(0.0)
    frames[fold["name"]] = d
    print(f"{fold['name']:<6} {len(d):>6,} merchants   "
          f"${d['realised_revenue'].sum()/1e6:>7,.1f}M realised in the target window   "
          f"{int((d['label_transactions'] == 0).sum()):>4} stopped trading")

A       3,994 merchants   $   18.1M realised in the target window     58 stopped trading


B       4,014 merchants   $   11.9M realised in the target window    110 stopped trading


C       4,022 merchants   $   14.9M realised in the target window     86 stopped trading


FINAL   4,025 merchants   $   16.5M realised in the target window     70 stopped trading


## 2. Fixed and learned comparisons

Fixed hand and Hybrid configurations come from the ranking notebook. Learned
non-negative coefficients are fitted on the preceding fold's feature/label pair.
For the FINAL evaluation (July-September), the latest fitting outcomes end in
June. FINAL still evaluates September, so subsequent model selection using this
comparison must not describe September as untouched. The separate frozen Machine
configuration is a historical fixed comparator, not a refit of these coefficients.


In [3]:
from sklearn.linear_model import LinearRegression
from scipy.stats import spearmanr



FEATURES = list(WEIGHTS_HAND)
SHRINK = {"estimated_bnpl_revenue", "log_value_growth"}
K = 300


def design(df):
    d = df.copy()
    d["stability"] = -d["value_cv"]
    d["customer_spread"] = 1 - d["customer_hhi"]
    shrinkage = d["transaction_count"] / (d["transaction_count"] + K)
    R = pd.DataFrame(
        {c: d[c].rank(pct=True, na_option="keep") * 100 for c in FEATURES}
    ).fillna(50.0)
    for c in SHRINK:
        R[c] = shrinkage * R[c] + (1 - shrinkage) * 50.0
    return R


def judge(frame, series):
    perfect = frame.nlargest(100, "realised_revenue")
    best = perfect["realised_revenue"].sum()
    sel = frame.assign(_s=series).nlargest(100, "_s")
    return {
        "spearman": spearmanr(series, frame["realised_revenue"])[0],
        "capture rate": sel["realised_revenue"].sum() / best,
        "hits": len(set(sel["merchant_abn"]) & set(perfect["merchant_abn"])),
        "stopped": int((sel["label_transactions"] == 0).sum()),
    }


rows, learned_by_fold = [], {}

for i, fold in enumerate(FOLDS):
    name = fold["name"]
    frame = frames[name]
    X = design(frame)

    rows.append({"fold": name, "picker": "hand weights",
                 **judge(frame, sum(X[c] * w for c, w in WEIGHTS_HAND.items()))})

    rows.append({"fold": name, "picker": "hybrid: evidence + ABS floor",
                 **judge(frame, sum(X[c] * w for c, w in WEIGHTS_HYBRID.items()))})

    if i > 0:  # fit on the previous fold's pair - it ends before this target begins
        prev = frames[FOLDS[i - 1]["name"]]
        model = LinearRegression(positive=True).fit(
            design(prev), np.log1p(prev["realised_revenue"]))
        coef = pd.Series(model.coef_, index=FEATURES)
        learned_by_fold[name] = coef / coef.sum() if coef.sum() > 0 else coef
        rows.append({"fold": name, "picker": "learned non-negative",
                     **judge(frame, pd.Series(model.predict(X), index=frame.index))})

    rows.append({"fold": name, "picker": "baseline: past revenue",
                 **judge(frame, frame["estimated_bnpl_revenue"].rank(pct=True) * 100)})

results = pd.DataFrame(rows)
results.round(3)

/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/scipy/optimize/_nnls.py:93: RuntimeWarning: divide by zero encountered in matmul
  x, rnorm, info = _nnls(A, b, maxiter)
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/scipy/optimize/_nnls.py:93: RuntimeWarning: overflow encountered in matmul
  x, rnorm, info = _nnls(A, b, maxiter)
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/scipy/optimize/_nnls.py:93: RuntimeWarning: invalid value encountered in matmul
  x, rnorm, info = _nnls(A, b, maxiter)
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/matthewwihartanto/Library/Python/3.10/lib/p

,fold,picker,spearman,capture rate,hits,stopped
0,A,hand weights,0.887,0.822,71,0
1,A,hybrid: evidence + ABS floor,0.881,0.830,72,0
2,A,baseline: past revenue,0.968,1.000,99,0
3,B,hand weights,0.895,0.844,70,0
4,B,hybrid: evidence + ABS floor,0.892,0.858,73,0
5,B,learned non-negative,0.903,0.871,75,0
6,B,baseline: past revenue,0.961,0.997,96,0
7,C,hand weights,0.908,0.878,76,0
8,C,hybrid: evidence + ABS floor,0.903,0.870,76,0
9,C,learned non-negative,0.905,0.907,80,0


## 3. Does the ordering of the three hold up?

The table below is the point of the whole notebook: the same comparison, four times, on four
different stretches of time.

Read the **spread** as carefully as the average. A picker that wins by eight points in one fold and
loses by six in another is not better — it is unstable, and an average hides that.

In [4]:
capture = results.pivot(index="picker", columns="fold", values="capture rate")
capture = capture[[f["name"] for f in FOLDS]]
capture["mean"] = capture.mean(axis=1)
capture["worst"] = capture.min(axis=1)

print("CAPTURE RATE per fold\n")
print((capture * 100).round(1).to_string())

spear = results.pivot(index="picker", columns="fold", values="spearman")[
    [f["name"] for f in FOLDS]]
spear["mean"] = spear.mean(axis=1)
print("\n\nSPEARMAN per fold\n")
print(spear.round(3).to_string())

hits = results.pivot(index="picker", columns="fold", values="hits")[
    [f["name"] for f in FOLDS]]
print("\n\nHITS in the realised top 100 (out of 100)\n")
print(hits.to_string())

CAPTURE RATE per fold

fold                              A     B     C  FINAL  mean  worst
picker                                                             
baseline: past revenue        100.0  99.7  99.8  100.0  99.9   99.7
hand weights                   82.2  84.4  87.8   84.9  84.8   82.2
hybrid: evidence + ABS floor   83.0  85.8  87.0   89.6  86.4   83.0
learned non-negative            NaN  87.1  90.7   92.3  90.0   87.1


SPEARMAN per fold

fold                              A      B      C  FINAL   mean
picker                                                         
baseline: past revenue        0.968  0.961  0.966  0.969  0.966
hand weights                  0.887  0.895  0.908  0.917  0.902
hybrid: evidence + ABS floor  0.881  0.892  0.903  0.911  0.897
learned non-negative            NaN  0.903  0.905  0.913  0.907


HITS in the realised top 100 (out of 100)

fold                             A     B     C  FINAL
picker                                               
baseline: p

### Did the learned weights themselves stay still?

Weights refitted in each fold are three independent answers to the same question. Weights that jump
around between folds are fitting noise, whatever their score; weights that keep roughly the same
shape are describing something real about merchants.

In [5]:
if learned_by_fold:
    w_folds = pd.DataFrame(learned_by_fold)
    w_folds["mean"] = w_folds.mean(axis=1)
    w_folds["hand"] = pd.Series(WEIGHTS_HAND)
    print("Learned non-negative weights, refitted per fold (normalised to sum to 1)\n")
    print(w_folds.round(3).sort_values("mean", ascending=False).to_string())
else:
    print("no fold had an earlier window to learn from")

Learned non-negative weights, refitted per fold (normalised to sum to 1)

                               B      C  FINAL   mean  hand
estimated_bnpl_revenue     0.736  0.681  0.714  0.710  0.70
active_month_ratio         0.180  0.217  0.230  0.209  0.02
stability                  0.015  0.059  0.034  0.036  0.03
customer_spread            0.053  0.009  0.012  0.025  0.05
log_value_growth           0.003  0.032  0.009  0.015  0.08
customer_irsad             0.012  0.000  0.001  0.004  0.02
customer_income            0.000  0.002  0.000  0.001  0.03
transactions_per_customer  0.000  0.000  0.000  0.000  0.04
repeat_rate                0.000  0.000  0.000  0.000  0.03


In [6]:
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
COLORS = {"hand weights": "#F58518",
          "hybrid: evidence + ABS floor": "#4C78A8",
          "learned non-negative": "#54A24B",
          "baseline: past revenue": "#8C8C8C"}

fold_names = [f["name"] for f in FOLDS]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

x = np.arange(len(fold_names))
width = 0.21
for j, picker in enumerate(COLORS):
    vals = [results.query("fold == @f and picker == @picker")["capture rate"].mean() * 100
            for f in fold_names]
    axes[0].bar(x + (j - 1.5) * width, vals, width=width, color=COLORS[picker], label=picker)
axes[0].set_xticks(x); axes[0].set_xticklabels(fold_names)
axes[0].set_ylabel("capture rate (%)"); axes[0].set_ylim(0, 108)
axes[0].set_title("Capture rate across four walk-forward windows")
axes[0].legend(frameon=False, fontsize=8, loc="lower left")

for picker, color in COLORS.items():
    sub = results[results["picker"] == picker].set_index("fold").reindex(fold_names)
    axes[1].plot(fold_names, sub["spearman"], marker="o", color=color, label=picker)
axes[1].set_ylabel("Spearman"); axes[1].set_title("Rank correlation across the same windows")
axes[1].legend(frameon=False, fontsize=8, loc="lower left")

plt.tight_layout(); plt.show()

/var/folders/09/xnwr_j_x37qfyk85k3lsbpf00000gn/T/ipykernel_27175/1449555849.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 4. How to report this

Write the sentence that the table actually supports, not the one we hoped for:

- **If the order of the three pickers is the same in all four folds**, say so plainly: *"the
  comparison was repeated across four walk-forward windows with three-month targets, and the
  ordering did not change."* That is a much stronger claim than any single holdout, and it is the
  version a marker cannot poke at.
- **If the order changes between folds**, the honest sentence is that the differences between these
  rankings are within the noise of this dataset — which, given that past revenue has been close to
  perfect everywhere, is a perfectly respectable finding.

**The decision this notebook is meant to settle**

Compare the hybrid against the learned weights across the four folds. If the gap is under about a
point, the floors on the ABS features and growth are cheap, the hybrid is the version to ship, and
the report can state its price exactly. If the gap is large, the floors are expensive and the group
has a real trade-off to argue rather than a free one.

**Limitations that stay true either way**

1. **Folds overlap.** The feature windows expand, so fold C contains fold B's data. The targets never
   overlap, which is what matters for leakage, but the folds are not four independent experiments.
2. **Fold A is the weakest.** Seven months of features is barely enough for `log_value_growth`
   (it needs six active months), and no model is scored there at all.
3. **One dataset, one firm, synthetic consumers.** Everything here describes this data, not BNPL
   in general.
4. **The fraud penalty is not applied in this notebook.** These are pre-risk rankings, exactly as
   in `merchant_ranking_hybrid.ipynb`.

In [7]:
OUT = PROCESSED / "rolling_validation_results.csv"
results.to_csv(OUT, index=False)

OUT_W = PROCESSED / "rolling_learned_weights.csv"
if learned_by_fold:
    pd.DataFrame(learned_by_fold).to_csv(OUT_W)
    print(f"written: {OUT.name}, {OUT_W.name}")
else:
    print(f"written: {OUT.name}")

winner = (results.groupby("picker")["capture rate"].mean().sort_values(ascending=False))
print("\nAverage capture rate across folds:")
print((winner * 100).round(1).to_string())
if {"hybrid: evidence + ABS floor", "learned non-negative"} <= set(winner.index):
    shared = [f["name"] for f in FOLDS if f["name"] != "A"]   # learned has no fold A
    sub = results[results["fold"].isin(shared)]
    gap = (sub.query("picker == 'learned non-negative'")["capture rate"].mean()
           - sub.query("picker == 'hybrid: evidence + ABS floor'")["capture rate"].mean())
    print(f"\nPrice of the ABS and growth floors, folds {', '.join(shared)}: "
          f"{gap * 100:+.2f} capture-rate points")


written: rolling_validation_results.csv, rolling_learned_weights.csv

Average capture rate across folds:
picker
baseline: past revenue          99.9
learned non-negative            90.0
hybrid: evidence + ABS floor    86.4
hand weights                    84.8

Price of the ABS and growth floors, folds B, C, FINAL: +2.55 capture-rate points
